# 01 - Descarga automática de fuentes a Parquet en Unity Catalog Volume

Notebook corregido para Databricks Free Edition.

Cambios aplicados:

- Se elimina `kagglehub` y `kagglesdk` para evitar el error `cannot import name get_web_endpoint`.
- Kaggle se descarga con la API clásica `kaggle`.
- Hugging Face se descarga con `datasets` en una versión compatible con `pyarrow` de Databricks.
- Los datasets descargados se guardan automáticamente como Parquet en el volumen de Unity Catalog.
- Se genera manifiesto de auditoría y dataset combinado para que Bronze lo consuma automáticamente.

Salidas principales:

- Raw Parquet por fuente y split: `/Volumes/<catalog>/<schema>/<volume>/sources/raw/...`
- Parquet canónico por dataset: `/Volumes/<catalog>/<schema>/<volume>/sources/processed/canonical/...`
- Dataset combinado: `/Volumes/<catalog>/<schema>/<volume>/sources/processed/combined/financial_sentiment_all`
- Manifiesto de auditoría: `<catalog>.<schema>.source_dataset_manifest`


In [0]:
# Databricks notebook source
# Instalación controlada para evitar conflictos entre kagglehub/kagglesdk y pyarrow/datasets.
# No se usa kagglehub. La descarga de Kaggle se realiza con la API clásica de Kaggle.

%pip uninstall -y kagglehub kagglesdk
%pip install -q --upgrade "datasets==2.19.2" "huggingface_hub>=0.23,<1.0" "kaggle==1.6.17" "fsspec>=2023.1.0,<2025.0.0"

# COMMAND ----------

dbutils.library.restartPython()


In [0]:
from datetime import datetime
from html import unescape
from pathlib import Path
import os
import re
import shutil
import unicodedata
import zipfile

import pandas as pd
from datasets import load_dataset
from pyspark.sql import functions as F

In [0]:
# Widgets del job.
# En Databricks Jobs estos valores se pueden sobreescribir como parámetros de tarea.

def ensure_widget(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
    except Exception:
        pass
    try:
        value = dbutils.widgets.get(name)
    except Exception:
        value = default
    return value


def as_bool(value: str, default: bool = False) -> bool:
    if value is None:
        return default
    return str(value).strip().lower() in {"true", "1", "yes", "y", "si", "sí"}


catalog = ensure_widget("catalog", "workspace")
schema = ensure_widget("schema", "financial_sentiment")
volume = ensure_widget("volume", "raw_landing")

enable_huggingface = as_bool(ensure_widget("enable_huggingface", "true"), True)
enable_kaggle = as_bool(ensure_widget("enable_kaggle", "true"), True)
force_refresh = as_bool(ensure_widget("force_refresh", "false"), False)
fail_if_no_sources = as_bool(ensure_widget("fail_if_no_sources", "true"), True)
fail_if_any_source_fails = as_bool(ensure_widget("fail_if_any_source_fails", "false"), False)

kaggle_username = ensure_widget("kaggle_username", "").strip()
kaggle_key = ensure_widget("kaggle_key", "").strip()
hf_token = ensure_widget("hf_token", "").strip() or None

download_batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"catalog={catalog}")
print(f"schema={schema}")
print(f"volume={volume}")
print(f"enable_huggingface={enable_huggingface}")
print(f"enable_kaggle={enable_kaggle}")
print(f"download_batch_id={download_batch_id}")


In [0]:
# Preparación de Unity Catalog y rutas del volumen.
# Si el schema o el volumen ya existen, estas instrucciones no los modifican.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
except Exception as exc:
    print(f"No se pudo crear el volumen automáticamente. Si ya existe o no tienes permisos, puedes ignorar esto. Detalle: {exc}")

volume_root = f"/Volumes/{catalog}/{schema}/{volume}"
raw_root = f"{volume_root}/sources/raw"
canonical_root = f"{volume_root}/sources/processed/canonical"
combined_root = f"{volume_root}/sources/processed/combined"
manifest_root = f"{volume_root}/sources/manifests/{download_batch_id}"
hf_cache_path = f"{volume_root}/cache/huggingface"
kaggle_download_root = f"{volume_root}/sources/raw/kaggle_downloads"
tmp_dir = f"{volume_root}/temp"

manifest_table = f"{catalog}.{schema}.source_dataset_manifest"
run_log_table = f"{catalog}.{schema}.pipeline_run_log"

for path in [raw_root, canonical_root, combined_root, manifest_root, hf_cache_path, kaggle_download_root, tmp_dir]:
    dbutils.fs.mkdirs(path)

# Redirect all HuggingFace and Python temp operations to volume (Serverless doesn't allow /local_disk0/ writes)
os.environ["HF_DATASETS_CACHE"] = hf_cache_path
os.environ["HF_HOME"] = f"{volume_root}/cache/huggingface_home"
os.environ["TMPDIR"] = tmp_dir
os.environ["TEMP"] = tmp_dir
os.environ["TMP"] = tmp_dir

print(f"volume_root={volume_root}")
print(f"canonical_root={canonical_root}")
print(f"tmp_dir={tmp_dir}")

In [0]:
# Utilidades de limpieza, normalización y escritura a Parquet.

POSITIVE_LABELS = {
    "positive", "mildly positive", "moderately positive", "strong positive",
    "bullish", "pos", "positivo", "1"
}
NEGATIVE_LABELS = {
    "negative", "mildly negative", "moderately negative", "strong negative",
    "bearish", "neg", "negativo", "-1"
}
NEUTRAL_LABELS = {"neutral", "neu", "neutro", "0"}

BASE_COLUMNS = ["dataset_id", "dataset_label", "source_platform", "split", "text", "label_normalized"]


def clean_text(value):
    if pd.isna(value):
        return None
    text = unescape(str(value))
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text or None


def normalize_label(value):
    if pd.isna(value):
        return None
    normalized = str(value).strip().lower()
    if normalized in POSITIVE_LABELS:
        return "positive"
    if normalized in NEGATIVE_LABELS:
        return "negative"
    if normalized in NEUTRAL_LABELS:
        return "neutral"
    return normalized


def safe_dataset_key(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]+", "_", value).strip("_").lower()


def safe_columns(pdf: pd.DataFrame) -> pd.DataFrame:
    out = pdf.copy()
    new_columns = []
    seen = set()
    for idx, col in enumerate(out.columns):
        candidate = safe_dataset_key(str(col)) or f"column_{idx}"
        original = candidate
        suffix = 1
        while candidate in seen:
            suffix += 1
            candidate = f"{original}_{suffix}"
        seen.add(candidate)
        new_columns.append(candidate)
    out.columns = new_columns
    return out


def spark_safe_pdf(pdf: pd.DataFrame, stringify: bool = False) -> pd.DataFrame:
    out = safe_columns(pdf)
    if stringify:
        for col in out.columns:
            out[col] = out[col].astype(str)
    return out


def write_spark_parquet_from_pandas(pdf: pd.DataFrame, path: str, mode: str = "overwrite", stringify: bool = False) -> int:
    rows = int(len(pdf))
    if rows == 0:
        print(f"Sin filas para escribir en {path}")
        return 0
    safe_pdf = spark_safe_pdf(pdf, stringify=stringify)
    sdf = spark.createDataFrame(safe_pdf)
    sdf.write.mode(mode).format("parquet").save(path)
    return rows


def compute_split_counts(total_rows: int):
    test_rows = round(total_rows * 0.2)
    validation_rows = round(total_rows * 0.2)
    train_rows = total_rows - test_rows - validation_rows
    if train_rows < 0:
        validation_rows = max(validation_rows + train_rows, 0)
        train_rows = total_rows - test_rows - validation_rows
    if train_rows < 0:
        test_rows = max(test_rows + train_rows, 0)
        train_rows = total_rows - test_rows - validation_rows
    return {"train": train_rows, "validation": validation_rows, "test": test_rows}


def derive_curated_splits(raw_split_frames):
    ordered_names = [s for s in ["train", "validation", "test"] if s in raw_split_frames]
    ordered_names += [s for s in raw_split_frames.keys() if s not in ordered_names]
    raw_df = pd.concat([raw_split_frames[s] for s in ordered_names], ignore_index=True, sort=False)
    raw_df = raw_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    counts = compute_split_counts(len(raw_df))
    train_end = counts["train"]
    validation_end = train_end + counts["validation"]
    return {
        "train": raw_df.iloc[:train_end].reset_index(drop=True),
        "validation": raw_df.iloc[train_end:validation_end].reset_index(drop=True),
        "test": raw_df.iloc[validation_end:].reset_index(drop=True),
    }


def first_existing_column(df: pd.DataFrame, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    lowered = {str(c).lower(): c for c in df.columns}
    for col in candidates:
        if str(col).lower() in lowered:
            return lowered[str(col).lower()]
    return None


In [0]:
# Estandarizadores por fuente.
# Todos producen el contrato canónico: dataset_id, dataset_label, source_platform, split, text, label_normalized.

def standardize_lwrf(df, split_name):
    text_col = first_existing_column(df, ["input", "text", "sentence", "Sentence", "Text"])
    label_col = first_existing_column(df, ["output", "label", "sentiment", "Sentiment"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no encontradas para lwrf42. Columnas disponibles: {list(df.columns)}")
    out = df.copy()
    out["dataset_id"] = "lwrf42/financial-sentiment-dataset"
    out["dataset_label"] = "lwrf42_financial_sentiment_dataset"
    out["source_platform"] = "huggingface"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


def standardize_kenpache_english(df, split_name):
    out = df.copy()
    language_col = first_existing_column(out, ["language", "lang"])
    if language_col is not None:
        out = out[out[language_col].astype(str).str.lower().isin(["en", "english"])]
    text_col = first_existing_column(out, ["sentence", "text", "Sentence", "Text"])
    label_col = first_existing_column(out, ["label", "sentiment", "Sentiment"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no encontradas para Kenpache. Columnas disponibles: {list(df.columns)}")
    out["dataset_id"] = "Kenpache/multilingual-financial-sentiment"
    out["dataset_label"] = "kenpache_multilingual_financial_sentiment_en"
    out["source_platform"] = "huggingface"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


def standardize_maguid(df, split_name):
    text_col = first_existing_column(df, ["text", "sentence", "Sentence", "Text", "headline", "content"])
    label_col = first_existing_column(df, ["polarity", "label", "sentiment", "Sentiment"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no encontradas para maguid28. Columnas disponibles: {list(df.columns)}")
    out = df.copy()
    out["dataset_id"] = "maguid28/combined_financial_phrasebank_twitter_news_sentiment"
    out["dataset_label"] = "maguid28_combined_financial_phrasebank_twitter_news_sentiment"
    out["source_platform"] = "huggingface"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


def standardize_kaggle(df, split_name):
    text_col = first_existing_column(df, ["Sentence", "sentence", "text", "Text", "headline", "news", "content"])
    label_col = first_existing_column(df, ["Sentiment", "sentiment", "label", "Label"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no encontradas para Kaggle. Columnas disponibles: {list(df.columns)}")
    out = df.copy()
    out["dataset_id"] = "sbhatti/financial-sentiment-analysis"
    out["dataset_label"] = "sbhatti_financial_sentiment_analysis"
    out["source_platform"] = "kaggle"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


In [0]:
# Lectura automática de Hugging Face y Kaggle.

from kaggle.api.kaggle_api_extended import KaggleApi

def load_huggingface_raw_splits(config):
    print(f"Descargando Hugging Face: {config['repo_id']}")
    
    # Use keep_in_memory=True to avoid disk caching issues in Serverless
    ds = load_dataset(
        config["repo_id"],
        cache_dir=f"{hf_cache_path}/{safe_dataset_key(config['key'])}",
        token=hf_token,
        verification_mode="no_checks",
        keep_in_memory=True,
    )

    if hasattr(ds, "items"):
        return {split_name: split_ds.to_pandas() for split_name, split_ds in ds.items()}

    return {config.get("default_split", "train"): ds.to_pandas()}


def get_kaggle_api():
    if not kaggle_username or not kaggle_key:
        raise RuntimeError(
            "Kaggle está habilitado, pero faltan credenciales. "
            "Configura los parámetros kaggle_username y kaggle_key en el Job, "
            "o ejecuta con enable_kaggle=false."
        )
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
    api = KaggleApi()
    api.authenticate()
    return api


def load_kaggle_raw_splits(config):
    print(f"Descargando Kaggle: {config['repo_id']}")
    api = get_kaggle_api()

    raw_dir = Path(kaggle_download_root) / config["key"]
    extract_dir = raw_dir / "extracted"

    if force_refresh and raw_dir.exists():
        shutil.rmtree(raw_dir)

    raw_dir.mkdir(parents=True, exist_ok=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    api.dataset_download_files(
        config["repo_id"],
        path=str(raw_dir),
        unzip=False,
        quiet=False,
    )

    zip_files = sorted(raw_dir.glob("*.zip"))
    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, "r") as zf:
            zf.extractall(extract_dir)

    candidates = []
    for pattern in ["*.csv", "*.CSV", "*.xlsx", "*.xls"]:
        candidates.extend(sorted(extract_dir.rglob(pattern)))
        candidates.extend(sorted(raw_dir.rglob(pattern)))

    # Eliminar duplicados conservando orden.
    seen = set()
    unique_candidates = []
    for path in candidates:
        if str(path) not in seen:
            seen.add(str(path))
            unique_candidates.append(path)

    if not unique_candidates:
        raise FileNotFoundError(f"No se encontraron CSV/XLSX dentro de la descarga de Kaggle: {raw_dir}")

    frames = []
    for file_path in unique_candidates:
        try:
            if file_path.suffix.lower() == ".csv":
                pdf = pd.read_csv(file_path)
            else:
                pdf = pd.read_excel(file_path)
            pdf["__source_file"] = str(file_path)
            frames.append(pdf)
            print(f"Archivo Kaggle leído: {file_path} filas={len(pdf)}")
        except Exception as exc:
            print(f"Archivo Kaggle omitido por error de lectura: {file_path}. Detalle: {exc}")

    if not frames:
        raise ValueError(f"No se pudo leer ningún archivo válido de Kaggle para {config['repo_id']}")

    return {config.get("default_split", "train"): pd.concat(frames, ignore_index=True, sort=False)}

In [0]:
# Configuración de fuentes.

source_configs = [
    {
        "key": "lwrf42_financial_sentiment_dataset",
        "repo_id": "lwrf42/financial-sentiment-dataset",
        "source_platform": "huggingface",
        "standardizer": standardize_lwrf,
        "enabled": enable_huggingface,
    },
    {
        "key": "kenpache_multilingual_financial_sentiment",
        "repo_id": "Kenpache/multilingual-financial-sentiment",
        "source_platform": "huggingface",
        "standardizer": standardize_kenpache_english,
        "enabled": enable_huggingface,
    },
    {
        "key": "maguid28_combined_financial_phrasebank_twitter_news_sentiment",
        "repo_id": "maguid28/combined_financial_phrasebank_twitter_news_sentiment",
        "source_platform": "huggingface",
        "standardizer": standardize_maguid,
        "enabled": enable_huggingface,
    },
    {
        "key": "sbhatti_financial_sentiment_analysis",
        "repo_id": "sbhatti/financial-sentiment-analysis",
        "source_platform": "kaggle",
        "standardizer": standardize_kaggle,
        "enabled": enable_kaggle,
        "default_split": "train",
    },
]

manifest_rows = []
combined_frames = []
failed_sources = []


def append_manifest(config, status, split_name="all", rows_written=0, raw_parquet_path=None, canonical_parquet_path=None, error_message=None):
    manifest_rows.append({
        "download_batch_id": download_batch_id,
        "dataset_key": config["key"],
        "dataset_id": config["repo_id"],
        "source_platform": config["source_platform"],
        "status": status,
        "split_name": split_name,
        "rows_written": int(rows_written or 0),
        "raw_parquet_path": raw_parquet_path,
        "canonical_parquet_path": canonical_parquet_path,
        "error_message": error_message,
        "event_ts": datetime.utcnow(),
    })


def process_source(config):
    if not config["enabled"]:
        append_manifest(config, "skipped", error_message="Fuente deshabilitada por parámetro del job")
        return

    print("=" * 100)
    print(f"Procesando fuente: {config['repo_id']} [{config['source_platform']}]")

    if config["source_platform"] == "huggingface":
        raw_splits = load_huggingface_raw_splits(config)
    elif config["source_platform"] == "kaggle":
        raw_splits = load_kaggle_raw_splits(config)
    else:
        raise ValueError(f"source_platform no soportado: {config['source_platform']}")

    # 1) Guardar raw parquet por split en el volumen.
    for split_name, raw_pdf in raw_splits.items():
        raw_path = f"{raw_root}/{config['source_platform']}/{config['key']}/{split_name}"
        rows = write_spark_parquet_from_pandas(raw_pdf, raw_path, mode="overwrite", stringify=True)
        append_manifest(config, "raw_parquet_saved", split_name, rows, raw_parquet_path=raw_path)
        print(f"Raw Parquet guardado: {raw_path} filas={rows}")

    # 2) Derivar splits train/validation/test comparables y estandarizar columnas.
    curated_splits = derive_curated_splits(raw_splits)
    canonical_frames = []

    for split_name, split_pdf in curated_splits.items():
        canonical_split = config["standardizer"](split_pdf, split_name)
        canonical_split["download_batch_id"] = download_batch_id
        canonical_split["source_record_hash"] = pd.util.hash_pandas_object(
            canonical_split[["dataset_label", "split", "text", "label_normalized"]].astype(str),
            index=False,
        ).astype(str)
        canonical_frames.append(canonical_split)

    canonical_pdf = pd.concat(canonical_frames, ignore_index=True, sort=False)
    canonical_pdf = canonical_pdf[canonical_pdf["text"].notna() & canonical_pdf["label_normalized"].notna()].copy()
    canonical_pdf = canonical_pdf[canonical_pdf["text"].str.len() > 0].copy()
    canonical_pdf = canonical_pdf.drop_duplicates(subset=["source_record_hash"])

    canonical_path = f"{canonical_root}/{config['key']}"
    rows = write_spark_parquet_from_pandas(canonical_pdf, canonical_path, mode="overwrite", stringify=False)
    append_manifest(config, "canonical_parquet_saved", "all", rows, canonical_parquet_path=canonical_path)
    combined_frames.append(canonical_pdf)
    print(f"Canonical Parquet guardado: {canonical_path} filas={rows}")


In [0]:
# Ejecución de descarga, escritura de manifiesto y registro de evidencia.

for config in source_configs:
    try:
        process_source(config)
    except Exception as exc:
        failed_sources.append((config["key"], str(exc)))
        append_manifest(config, "failed", config.get("default_split", "all"), error_message=str(exc))
        print(f"Falló fuente {config['repo_id']}: {exc}")

successful_frames = [df for df in combined_frames if len(df) > 0]

if successful_frames:
    combined_pdf = pd.concat(successful_frames, ignore_index=True, sort=False)
    combined_pdf = combined_pdf.drop_duplicates(subset=["source_record_hash"])
    combined_path = f"{combined_root}/financial_sentiment_all"
    combined_rows = write_spark_parquet_from_pandas(combined_pdf, combined_path, mode="overwrite", stringify=False)

    dbutils.fs.rm(f"{manifest_root}/latest_successful_combined_path.txt", True)
    dbutils.fs.put(f"{manifest_root}/latest_successful_combined_path.txt", combined_path, True)
    print(f"Dataset combinado guardado: {combined_path} filas={combined_rows}")
else:
    combined_rows = 0
    if fail_if_no_sources:
        raise RuntimeError(
            "No se logró descargar ninguna fuente. "
            "Revisa conectividad, credenciales de Kaggle o restricciones de Databricks Free Edition."
        )

manifest_pdf = pd.DataFrame(manifest_rows)
if len(manifest_pdf) > 0:
    spark.createDataFrame(manifest_pdf).write.mode("append").format("delta").saveAsTable(manifest_table)
    # Write JSON directly to volume (Serverless doesn't allow /tmp/ access)
    manifest_json_str = manifest_pdf.to_json(orient="records", date_format="iso", force_ascii=False)
    dbutils.fs.put(f"{manifest_root}/{download_batch_id}_manifest.json", manifest_json_str, True)

if failed_sources and fail_if_any_source_fails:
    raise RuntimeError(f"Fallaron fuentes requeridas: {failed_sources}")

status = "SUCCESS" if not failed_sources else "PARTIAL_SUCCESS"
message = (
    f"Descarga automática finalizada. "
    f"Fuentes exitosas={len(successful_frames)}, fallidas={len(failed_sources)}, filas combinadas={combined_rows}."
)

spark.createDataFrame([
    (download_batch_id, "01_download_source_parquets", status, message, int(combined_rows), datetime.utcnow())
], "run_id string, step string, status string, message string, rows_written long, event_ts timestamp") \
    .write.mode("append").format("delta").saveAsTable(run_log_table)

try:
    dbutils.jobs.taskValues.set(key="download_batch_id", value=download_batch_id)
    dbutils.jobs.taskValues.set(key="canonical_root", value=canonical_root)
    dbutils.jobs.taskValues.set(key="combined_path", value=f"{combined_root}/financial_sentiment_all")
    dbutils.jobs.taskValues.set(key="combined_rows", value=str(combined_rows))
except Exception:
    pass

display(spark.table(manifest_table).where(F.col("download_batch_id") == download_batch_id))
print(message)